In [ ]:
# Resolve paths consistently from either the repository root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src').is_dir() and (p / 'data').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from within the cloned repository.')
os.chdir(PROJECT_ROOT / 'notebooks')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Identifying and fixing missing reports of crime in USA

In [ ]:
#sets the notebook width
from IPython.display import display, HTML
display(HTML("<style>.container { width:85% !important; }</style>"))
%load_ext autoreload
%autoreload 2

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Standard Libraries
import sys
import os
from pathlib import Path
import glob
import math

# Third party libraries
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

import statsmodels.api as sm
# from pymc3 import Normal, Model, Uniform,Poisson,Deterministic
# from pymc3 import traceplot,forestplot,summary
# from pymc3 import glm, Model, Metropolis, NUTS, sample
# import pymc3 as pm

# figures
import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker

# Local
# from modules import ste

In [ ]:
import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker

def stdfigsize(scale=1, nx=1, ny=1, ratio=1.3):
    """
    Returns a tuple to be used as figure size.
    -------
    returns (7*ratio*scale*nx, 7.*scale*ny)
    By default: ratio=1.3
    If ratio<0 them ratio = golden ratio
    """
    if ratio < 0:
        ratio = 1.61803398875
    return((7*ratio*scale*nx, 7*scale*ny))

def stdrcparams(usetex=False):
    """
    Set several mpl.rcParams and sns.set_style for my taste.
    ----
    usetex = True
    ----
    """
    sns.set_style("white")
    sns.set_style({"xtick.direction": "in",
                 "ytick.direction": "in"})
    rcparams = {'text.usetex': usetex,
              'font.family': 'sans-serif',
              'font.sans-serif': ['Times'],
#               'text.latex.unicode': True,
#               'text.latex.preamble': [r"\usepackage[T1]{fontenc}",
#                                       r"\usepackage{lmodern}",
#                                       r"\usepackage{amsmath}",
#                                       r"\usepackage{mathptmx}"
#                                       ],
              'axes.labelsize': 30,
              'axes.titlesize': 30,
              'axes.spines.left':True,
              'axes.spines.bottom':True,
              'axes.spines.top': False,
              'axes.spines.right':False,
              'ytick.left': 'on',
              'xtick.bottom': 'on',
              'xtick.labelsize': '25',
              'ytick.labelsize': '25',
              'axes.linewidth': 1.8,
              'xtick.major.width': 1.8,
              'xtick.minor.width': 1.8,
              'xtick.major.size': 14,
              'xtick.minor.size': 7,
              'xtick.major.pad': 10,
              'xtick.minor.pad': 10,
              'ytick.major.width': 1.8,
              'ytick.minor.width': 1.8,
              'ytick.major.size': 14,
              'ytick.minor.size': 7,
              'ytick.major.pad': 10,
              'ytick.minor.pad': 10,
              'axes.labelpad': 15,
              'axes.titlepad': 15,
              "xtick.direction":
                "out",
              "ytick.direction": "out"
              }
    mpl.rcParams.update(rcparams)

mpl.rcParams['lines.linewidth'] = 5
mpl.rcParams['lines.color'] = '#3690c0'

In [ ]:
stdrcparams(usetex=False)
figsize=stdfigsize(ratio=-1)
xs,ys=figsize
figsize

# Data

## Load demographic and socieconomic data

In [ ]:
full_data = pd.read_csv('../data/clean_dataframes/urban_metrics_interpolated_and_cleaned_1960_2020.csv', index_col=0)
full_data.head()

## Crime data

In [ ]:
crime_county=pd.read_csv('../data/clean_dataframes/crime_county_data_extended.csv',index_col=0)
crime_county.head()

# Fix county names 

In [ ]:
full_data.County=full_data.County+', '+full_data.State

In [ ]:
full_data.County=full_data.County.str.replace(' County','')
full_data.County=full_data.County.str.strip()
full_data.head()

In [ ]:
crime_county.County=crime_county.County+', '+crime_county.State
crime_county.head()

In [ ]:
crime_county['FIPS']=crime_county['FIPS'].astype(str)
crime_county.head()

# Merge

In [ ]:
data=pd.merge(crime_county,full_data,on=['County','State','Year'],how='outer')
# data.sort_values(['State','FIPS','Year'],inplace=True)
data=data[np.isnan(data.Population)==False]
# data.drop_duplicates()
data.reset_index(drop=True,inplace=True)
data.head()

In [ ]:
pairs=[data.CountyFIPS,data.County]

In [ ]:
data['FIPS_State']=data["CountyFIPS"].map(str) +', '+ data["State"]

In [ ]:
df=data.copy()
df=df[np.isnan(df.MURDER)==False].reset_index(drop=True)

In [ ]:
df=df[['County','FIPS_State']].drop_duplicates().reset_index(drop=True)

In [ ]:
countyfipmap=dict(np.transpose([df['FIPS_State'].values,df['County'].values]))

In [ ]:
list_of_values=list(countyfipmap.keys())
data=data[data['FIPS_State'].isin(list_of_values)].reset_index(drop=True)

In [ ]:
x=np.array(data.FIPS_State)
mmap=[countyfipmap[item] for item in x]

In [ ]:
data['County']=mmap

In [ ]:
data[(data.FIPS=='12086')&(data.State=='FL')].head()

In [ ]:
data[data.State=='NE']

In [ ]:
data.rename(columns={'MURDER':'Murder'}, inplace = True)

In [ ]:
fig = plt.figure(figsize=(xs*2,ys*3))
variable=[column for column in ['Murder', 'Robbery',
       'Assault', 'Burglary', 'Larceny', 'Motor Vehicle Theft'] if column in data.columns]
for j,name in enumerate(variable):
    ax = fig.add_subplot(4,2,j+1)
    
    df=data[(data.County=='Cook, IL')]
    x=df.Year
    y=df[variable[j]]
    ax.plot(x,y,color='k',linewidth=2,markersize=12,alpha=0.6)
    ax.set_ylabel(r"{}".format(variable[j]),multialignment='center')
    ax.set_xlabel(r"Year, $t$")
plt.tight_layout()

# Removing negative values

In [ ]:
data.columns

In [ ]:
for var in ['Population', 'Year', 'Whites', 'Poverty', 'Blacks', 'Minorities','Non-White','Education', 'Unemployment']:
    data.loc[data[var]<0, var] = np.nan

In [ ]:
data.head()

In [ ]:
data[data.County=='Miami-Dade, FL']

In [ ]:
df=data[['FIPS','State']].copy()

county_state=df.drop_duplicates().reset_index(drop=True)

In [ ]:
df=data[['County','FIPS','State']]
df=df.drop_duplicates().reset_index(drop=True)

In [ ]:
crime_county[crime_county['County'].str.contains("COUNTY")]

In [ ]:
df=data[(data['CountyFIPS']==79)&(data['State']=='WI')]    
df.head()

In [ ]:
county_state

In [ ]:
for county,state in county_state.values[:3]:
    print(county,state)
    df=data[(data['FIPS']==county)&(data['State']==state)]    
    fig, ax = plt.subplots(figsize=(xs,ys/2))
    y=list(df['Murder'].rolling(3).mean()[1:])
    y.append(np.nan)
    ax.plot(df.Year,y,'k-',linewidth=3)
    ax.bar(df.Year.values,df.Murder.values,width=1,color='#2b8cbe')

    ax.set_ylabel(r"Murders",multialignment='center')
    ax.set_xlabel(r"Year")
    ax.set_title(df.County.values[0])

    plt.show()


In [ ]:
data.head()

# Save data

In [ ]:
data['Year'] = data['Year'].astype(int)
data.head()

In [ ]:
data=data[['County', 'FIPS', 'State', 'Year', 'Murder',
       'Population', 'Whites', 'Poverty', 'Blacks', 'Minorities','Non-White', 'Education', 'Unemployment']]
data.head()

In [ ]:
df=data[data.County=='Milwaukee, WI']

In [ ]:
data[data.County=='Miami-Dade, FL'].head()

In [ ]:
df.head()

In [ ]:
data.head()

In [ ]:
data[data.County=='Abbeville, SC'].head()

In [ ]:
data[data.State=='NE']

In [ ]:
data[data.County.str.contains('San Fran')].head()

In [ ]:
data.head()

In [ ]:
data.to_csv('../data/clean_dataframes/full_merged_data_1960_2020.csv')

In [ ]:
len(np.unique(data[~data['Unemployment'].isna()]['County']))